# 01 — Data collection & exploration

**Owners:** game-side tables: minhnguyen1210 · player-side tables: CaoThien1402

Spec sections 3.1 (data collection) and 3.2 (data exploration).

**Inputs:** TODO: files this notebook reads (`data/raw/<platform>/<table>.csv` or `data/processed/...`).

**Outputs:** TODO: files this notebook writes to `data/processed/`, and figures saved to `reports/figures/`.

In [1]:
import sys
from pathlib import Path

# Make the repo root importable (works from the repo root or from notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RAW_DIR, PROCESSED_DIR, FIG_DIR, PLATFORMS, TABLES, RANDOM_STATE
from src.data_loading import load_table, load_all
from src.utils import report_rows

pd.set_option("display.max_columns", 50)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1 Data collection

### 1.1 Subject

TODO: what topic does the data cover, and what real-world context does it represent?

### 1.2 Source

TODO: platform and full URL, original author or organization, publication date (we use version 1).

### 1.3 License

TODO: which license applies; restrictions or attribution requirements.

### 1.4 Collection method

TODO: how the data was collected, from which population, over what period, known limitations and biases.

### 1.5 Motivation

TODO: why our group chose this dataset.

## 2 Data exploration

Go deep on the columns relevant to our questions; summarise the rest briefly instead of one chart per column.

Each subsection has one code cell per table group, so the two members can work in parallel:
**game side** (`games`, `achievements`, `prices`) and **player side** (`players`, `history`, `purchased_games`,
`reviews`, `friends`, `private_steamids`).

### 2.1 Loading strategy

**Player-side tables** (`players`, `history`, `purchased_games`, `reviews`, `friends`, `private_steamids`; owner: CaoThien1402).
The game-side tables (`games`, `achievements`, `prices`) are covered by minhnguyen1210 in the first code cell of each subsection.

All player-side tables fit in memory (the largest, PlayStation `history`, needs 1.6 GB), so nothing is read in chunks
for now. The settings live in `PLAYER_READ_ARGS` in `src/data_loading.py` and are used through `load_player_table`.

| Decision | Tables | Why |
| --- | --- | --- |
| `int64` for Steam ids, `int32` for PlayStation and Xbox ids | all | Steam ids are about 7.6e16 and overflow `int32`; the other platforms use ids below 2^31 |
| `parse_dates` | `history.date_acquired`, `players.created`, `reviews.posted` | dates are read as text otherwise, so time arithmetic and resampling need a conversion later |
| `country` as `category` | Steam, PlayStation `players` | few distinct values repeated over hundreds of thousands of rows: the Steam table drops from 56 MB to 8 MB |
| `achievementid` as `string` | `history` | the id is text (`403640_ACH_1`); `history` uses 42-47% less memory than with the defaults (measured on 200,000 rows) |
| `usecols` without `review` | `reviews` | the review text is about 94% of the memory (715 MB with the text, 43 MB without it); Q4 reads the text only if it builds text features |
| `keep_default_na=False, na_values=[""]` | `players` | only an empty cell is missing; with the default setting pandas also turns the nickname `null` (one Xbox player) into `NaN` |
| one table at a time, then `del` | `history`, `reviews`, `friends` | keeps memory below about 2 GB; reading `history` takes 20-33 s per platform |

**Columns that hold a list as text.** `library` (`purchased_games`) and `friends` (`friends`) store lists such as
`"[60, 1670]"`. Parsing about 65 million ids with `ast.literal_eval` would be slow, so the tables are kept as text and
`count_list_items` counts the items (number of commas plus one). `parse_id_list` turns a single cell into a Python list
when the ids themselves are needed (for example for `explode`).

**Empty list cells are real missing data, not an empty list.** In the raw file, 55,607 of 102,548 Steam libraries
(54.2%) and 85,222 of 424,683 friend lists (20.1%) are empty cells; the text `"[]"` never appears. PlayStation and Xbox
libraries have no empty cell. Most likely these are private profiles, but the dataset does not say so; we treat them as
unknown, not as "owns no game". Any statement about library size or friends is therefore limited to the players who
made them public.

**Other things checked while loading**
- `reviews` contains text cells that span several lines. The default C parser of pandas reads them correctly (1,204,534
  rows, the same number as the author's description), so we do not use the `pyarrow` engine for this table.
- Steam `players.country` has 177,868 empty cells (41.9%); PlayStation `players` has a `country` column, Xbox has none.
  See section 2.7.
- `private_steamids` has no duplicated id (227,963 rows).

The two tables below show the result for every player-side table: size on disk, rows, memory after loading, saving
from the dtype settings on a 200,000-row sample, and read time. `reviews` is shown with its text (see above).

In [2]:
# Game side: games, achievements, prices (owner: TODO)

In [3]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: CaoThien1402)
import gc
import time

from src.data_loading import PLAYER_SIDE_TABLES, count_list_items, load_player_table, table_path

SAMPLE_ROWS = 200_000  # sample used to compare memory with and without the settings in PLAYER_READ_ARGS

records = []
for platform, name in PLAYER_SIDE_TABLES:
    # memory saved by dtype / parse_dates: same first rows, default settings vs PLAYER_READ_ARGS
    default_mb = load_table(platform, name, nrows=SAMPLE_ROWS).memory_usage(deep=True).sum() / 1e6
    tuned_mb = load_player_table(platform, name, nrows=SAMPLE_ROWS).memory_usage(deep=True).sum() / 1e6

    start = time.perf_counter()
    df = load_player_table(platform, name)  # the full table, one at a time
    seconds = time.perf_counter() - start
    records.append({
        "platform": platform, "table": name,
        "file_MB": round(table_path(platform, name).stat().st_size / 1e6),
        "rows": len(df), "columns": df.shape[1],
        "memory_MB": round(df.memory_usage(deep=True).sum() / 1e6),
        "sample_saving_%": round(100 * (1 - tuned_mb / default_mb)),
        "seconds": round(seconds, 1),
    })
    del df
    gc.collect()

loading_summary = pd.DataFrame(records)
loading_summary

,platform,table,file_MB,rows,columns,memory_MB,sample_saving_%,seconds
0,steam,players,18,424683,3,8,86,0.8
1,playstation,players,11,356600,3,26,49,0.3
2,xbox,players,5,274450,2,20,5,0.2
3,steam,history,648,10693879,3,1001,42,20.5
4,playstation,history,799,19510083,3,1571,47,32.8
5,xbox,history,633,15275900,3,1239,47,24.3
6,steam,purchased_games,92,102548,2,95,0,1.2
7,playstation,purchased_games,74,46582,2,77,0,0.9
8,xbox,purchased_games,86,46466,2,89,0,1.1
9,steam,reviews,552,1204534,8,715,13,11.8


In [4]:
# Columns that store a list as text: how many cells are empty, and how many ids do they hold?
list_columns = [("steam", "purchased_games", "library"), ("playstation", "purchased_games", "library"),
                ("xbox", "purchased_games", "library"), ("steam", "friends", "friends")]
rows = []
for platform, name, column in list_columns:
    df = load_player_table(platform, name)
    items = count_list_items(df[column])
    rows.append({"platform": platform, "table": name, "rows": len(df),
                 "empty_cells": int(df[column].isna().sum()),
                 "empty_%": round(100 * df[column].isna().mean(), 1),
                 "ids_in_lists": int(items.sum()), "max_per_row": int(items.max())})
    del df
pd.DataFrame(rows)

,platform,table,rows,empty_cells,empty_%,ids_in_lists,max_per_row
0,steam,purchased_games,102548,55607,54.2,11258715,32463
1,playstation,purchased_games,46582,0,0.0,10893552,13540
2,xbox,purchased_games,46466,0,0.0,12667877,9018
3,steam,friends,424683,85222,20.1,30903334,2000


### 2.2 Overview & data dictionary

TODO, per table used:
- number of rows and columns; what one row represents;
- meaning and data type of each column; columns to drop or convert;
- duplicated or empty rows, and whether to remove them (justify).

| Table | Column | Type | Meaning | Action (keep / convert / drop) |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

In [5]:
# Game side: games, achievements, prices (owner: TODO)

In [6]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.3 Keys & joins between tables

TODO: for every join the questions will need (e.g. `history` → `achievements` → `games`, `prices` → `games`,
`reviews` → `players`): rows before and after, and the number of keys without a match on each side.

In [7]:
# Game side: games, achievements, prices (owner: TODO)

In [8]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.4 Time coverage

TODO: for every date column, per platform: first and last date, frequency, gaps; suspicious dates (placeholders,
spikes); how the end of data collection limits recent periods.

In [9]:
# Game side: games, achievements, prices (owner: TODO)

In [10]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.5 Numerical columns

TODO: distribution shape, centre and spread; plausible min/max; outliers (IQR, z-score) genuine or errors;
impossible or placeholder values.

In [11]:
# Game side: games, achievements, prices (owner: TODO)

In [12]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.6 Categorical columns

TODO: number of distinct values and most frequent ones; balance; inconsistent labels and typos;
rare categories to group.

In [13]:
# Game side: games, achievements, prices (owner: TODO)

In [14]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.7 Missing data

TODO: count and percentage per column; whether missingness follows a pattern; why values are missing;
how each column will be handled (drop, impute, separate category).

In [15]:
# Game side: games, achievements, prices (owner: TODO)

In [16]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.8 Relationships

TODO: correlations between numerical variables; cross-tabulations and grouped statistics for important
combinations; surprising relationships.

In [17]:
# Game side: games, achievements, prices (owner: TODO)

In [18]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

## 3 Exploration summary

### 3.1 Key observations (3–5)

TODO

### 3.2 Serious data quality issues

TODO

### 3.3 Preprocessing plan

TODO: the cleaning steps shared by several questions, and the files they produce.

| Step | Tables | Rule | Output in `data/processed/` | Used by |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

### 3.4 Patterns that could lead to questions

TODO